<a href="https://colab.research.google.com/github/MAHMOUD-lab26/MachineLearning2/blob/main/ML_sem_5_proj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
# Load the training data
train_df = pd.read_csv('/content/train.csv')

# Define features (X) and target (y)
X = train_df.drop('Usage_kWh', axis=1)
y = train_df['Usage_kWh']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Data loaded and split successfully.")

Data loaded and split successfully.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer # Import SimpleImputer

# Identify categorical and numerical features
categorical_features = ['WeekStatus', 'Day_of_week', 'Load_Type']
numerical_features = ['Lagging_Current_Reactive.Power_kVarh', 'Leading_Current_Reactive_Power_kVarh', 'CO2(tCO2)', 'Lagging_Current_Power_Factor', 'Leading_Current_Power_Factor', 'NSM']

# Create a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='mean')), ('scaler', StandardScaler())]), numerical_features), # Add SimpleImputer to numerical features
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough' # Keep other columns (like 'date' and 'Id')
)

# Create the Ridge regression pipeline
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('ridge', Ridge(alpha=0.1))])

# Train the pipeline
# Remove the 'date' column before training
pipeline.fit(X_train.drop('date', axis=1), y_train)

# Make predictions on the test data
# Remove the 'date' column before predicting
predictions = pipeline.predict(X_test.drop('date', axis=1))

# Create a DataFrame with the predictions and 'Id' from the test set
output_df_tuned_pipeline = pd.DataFrame({'Id': X_test['Id'], 'Usage_kWh_Predicted_Tuned_Pipeline': predictions})

# Display the first few rows of the output DataFrame
display(output_df_tuned_pipeline.head())

,Id,Usage_kWh_Predicted_Tuned_Pipeline
21957,21958,44.591177
5034,5035,83.069704
21313,21314,1.239562
22979,22980,45.646184
18742,18743,1.873323


In [ ]:
from sklearn.metrics import r2_score

# Calculate R2 score on the training data
r2_train_tuned_pipeline = r2_score(y_train, pipeline.predict(X_train))

# Print the R2 score
print(f"R2 score on training data (tuned pipeline): {r2_train_tuned_pipeline}")

# Save the predictions to a CSV file
output_df_tuned_pipeline.to_csv('prediction_Ridge.csv', index=False)

R2 score on training data (tuned pipeline): 0.9783170209980835


In [ ]:
# Load the test data
test_df = pd.read_csv('/content/test.csv')

# Make predictions on the test data
# Remove the 'date' column before predicting, similar to how it was done for training
test_predictions = pipeline.predict(test_df.drop('date', axis=1))

# Create a DataFrame with the 'Id' from the test set and the predictions
submission_df = pd.DataFrame({'Id': test_df['Id'], 'Usage_kWh': test_predictions})

# Save the submission DataFrame to a CSV file
submission_df.to_csv('submission.csv', index=False)

print("Submission file created successfully: submission.csv")
display(submission_df.head())

Submission file created successfully: submission.csv


,Id,Usage_kWh
0,28000,107.630183
1,28001,88.251372
2,28002,88.536672
3,28003,65.846939
4,28004,65.536189
